<a href="https://colab.research.google.com/github/mayait/CursoAnalisisDatos_IA_2026/blob/main/sitio/labs/lab_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Laboratorio 3 · Estadística descriptiva sin mentir

La semana pasada decidiste la unidad de análisis y viste que «cuántas ventas hicimos» tiene tres
respuestas. Hoy la pregunta es otra: cuando ya sabes qué cuentas, ¿qué número lo resume? El promedio
es la estadística más usada del mundo y la que más decisiones arruina. La industria de esta semana es
la **música**: 32 833 canciones del catálogo de Spotify con su duración, su popularidad y su género.
Vas a contestar dos preguntas que cualquier sello discográfico paga por saber —*¿qué tan larga es una
canción típica?* y *¿qué tan popular es nuestro catálogo?*— y vas a descubrir que la media miente en
las dos, cada vez en una dirección distinta. Esto desbloquea la línea base descriptiva contra la que
se compara todo lo que viene después.

> **Hoy haces** · Perfilas el catálogo completo con `describe`, `quantile` y `value_counts`
> (90 min). Mides duración y popularidad con media y mediana, lees percentiles como umbrales de
> negocio, cuentas géneros con `value_counts` y comparas dispersión con el coeficiente de variación.
> Cierras con doce preguntas descriptivas contra reloj.
>
> **Entrega** · Este cuaderno ejecutado, los tres ejercicios resueltos y las doce preguntas
> respondidas, cada una indicando si el estadístico correcto es la media o la mediana y por qué.
> Nombre de archivo: `lab_03_apellido.ipynb`.

In [ ]:
# --- Setup del entorno ---
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# Los datasets públicos de las semanas 1 a 4 viven en sitio/datos/publicos/
CANDIDATOS = [Path("../datos/publicos"), Path("datos/publicos"), Path("sitio/datos/publicos")]
DATOS = next((p for p in CANDIDATOS if p.exists()), None)
if DATOS is None:
    raise FileNotFoundError(
        "No encuentro la carpeta de datos. En Colab ejecuta primero:\n"
        "  !git clone https://github.com/<usuario>/CursoAnalisisDatos_IA_2026.git")

print("Setup completo ✓")
print(f"pandas {pd.__version__} · datos en {DATOS.resolve()}")

## 1. Preparación mínima

Antes de describir hay que saber qué se describe. Hoy se hacen dos cosas y ninguna más: convertir la
duración de milisegundos a minutos —nadie negocia en milisegundos— y dejar anotada la unidad de
análisis, porque este archivo tiene la misma trampa que Superstore: **una fila no es una canción, es
una aparición de una canción en una playlist**. La misma canción puede estar en varias. Hoy trabajas
sobre filas y lo dices; al final del cuaderno verás cuánto cuesta olvidarlo.

In [ ]:
canciones = pd.read_csv(DATOS / "spotify_canciones.csv")
canciones["duracion_min"] = canciones["duration_ms"] / 60_000

print(f"filas (apariciones en playlists) : {len(canciones):,}")
print(f"canciones únicas (track_id)      : {canciones['track_id'].nunique():,}")
print(f"artistas distintos               : {canciones['track_artist'].nunique():,}")
print(f"playlists                        : {canciones['playlist_id'].nunique():,}")
print(f"géneros de playlist              : {canciones['playlist_genre'].nunique()}")

## 2. `describe`: el perfilado que cabe en una línea

`describe()` sobre el DataFrame entero es el primer gesto de cualquier perfilado. Con
`include="all"` incluye también las columnas de texto, y ahí aparecen `count`, `unique`, `top` y
`freq`, que son las que sirven para categorías.

In [ ]:
canciones[["duracion_min", "track_popularity", "danceability", "energy", "tempo"]].describe()

In [ ]:
canciones.describe(include="all").T

Léelo por filas, no por columnas. `duracion_min` va de 0,07 a 8,63 minutos: hay algo de cuatro
segundos vendido como canción, y eso es una pregunta, no un error de pandas. `track_popularity` va
de 0 a 100 con la media en 42,48. Y `count` no es igual en todas las columnas: `track_name` y
`track_artist` tienen 5 huecos. Todo lo que llame la atención se anota; la semana 4 es para
perseguirlo.

## 3. La duración: ¿qué tan larga es una canción típica?

La pregunta parece trivial y mueve dinero: las radios programan bloques, las playlists editoriales
tienen límites tácitos y la regalía por reproducción no distingue un tema de 2 minutos de uno de 8.
Empieza por los dos resúmenes clásicos y su diferencia.

In [ ]:
d = canciones["duracion_min"]

print(f"media   : {d.mean():.2f} minutos")
print(f"mediana : {d.median():.2f} minutos")
print(f"desviación estándar : {d.std():.2f}")
print(f"canciones por debajo de la media : {(d < d.mean()).mean():.1%}")
print(f"canciones de entre 3 y 4 minutos : {((d >= 3) & (d < 4)).mean():.1%}")
print()
larga = canciones.loc[d.idxmax()]
corta = canciones.loc[d.idxmin()]
print(f"la más larga : {larga['track_name']} — {larga['track_artist']} ({larga['duracion_min']:.2f} min)")
print(f"la más corta : {corta['track_name']} — {corta['track_artist']} ({corta['duracion_min'] * 60:.0f} segundos)")

In [ ]:
fig, ax = plt.subplots()
ax.hist(d, bins=80, color="#4C72B0", edgecolor="white", linewidth=0.4)
ax.axvline(d.median(), color="#55A868", linewidth=2.5, label=f"mediana {d.median():.2f}")
ax.axvline(d.mean(), color="#C44E52", linewidth=2.5, linestyle="--", label=f"media {d.mean():.2f}")
ax.set_title("La canción típica dura 3,6 minutos: la cola de temas largos arrastra la media hacia arriba")
ax.set_xlabel("duración (minutos)")
ax.set_ylabel("número de canciones")
ax.legend()
plt.tight_layout()
plt.show()

📌 **Media 3,76 y mediana 3,60.** La separación es pequeña pero sistemática: la cola derecha —temas
de 6, 7 y 8 minutos— tira de la media, y el 57,9 % de las canciones queda por debajo de ella. Con
esta forma, la mediana es la respuesta honesta a «¿qué tan larga es una canción típica?»: 3 minutos
36 segundos. Y el dato que un programador de radio quiere de verdad no es ninguno de los dos: es que
**casi la mitad del catálogo (48,7 %) cae entre 3 y 4 minutos**. El formato canción existe y se ve en
un histograma.

## 4. Percentiles: `quantile` y su lectura de negocio

Un percentil no es un adorno estadístico: es un umbral operativo. El p90 de la duración contesta
«¿desde qué largo una canción está en el 10 % más extenso?», que es exactamente la pregunta que hace
falta para poner una regla de corte en una playlist editorial.

In [ ]:
cortes = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
rock = canciones.loc[canciones["playlist_genre"] == "rock", "duracion_min"]
rap = canciones.loc[canciones["playlist_genre"] == "rap", "duracion_min"]

percentiles = pd.DataFrame({
    "catálogo completo": d.quantile(cortes),
    "rock": rock.quantile(cortes),
    "rap": rap.quantile(cortes),
})
percentiles.index = [f"p{int(c * 100):02d}" for c in cortes]
percentiles

Tres lecturas de negocio, una por columna:

- **p90 global = 5,00.** Una regla del tipo «nada de más de cinco minutos» rechaza exactamente al
  10 % del catálogo. Ahora la regla tiene precio.
- **p99 global = 7,17.** Un tema de más de siete minutos es más raro que 99 de cada 100. Si tu
  banda insiste en grabarlo, que sea por arte y no por estrategia.
- **Rock corre más largo que rap en todo el recorrido**: su mediana (3,96) roza el p75 del
  catálogo completo. Comparar percentil contra percentil, no solo medias, es lo que evita disparates
  como pedirle al rock canciones de radio de tres minutos.

## 5. La popularidad: la media que miente hacia abajo

`track_popularity` va de 0 a 100. Con la duración, la cola larga estaba a la derecha y la media
quedaba **por encima** de la mediana. Aquí ocurre lo contrario, y la causa tiene nombre de negocio.

In [ ]:
pop = canciones["track_popularity"]

print(f"media   : {pop.mean():.2f}")
print(f"mediana : {pop.median():.0f}")
print(f"canciones con popularidad exactamente 0 : {(pop == 0).sum():,} ({(pop == 0).mean():.1%})")
print(f"media sin los ceros    : {pop[pop > 0].mean():.2f}")
print(f"canciones con 80 o más : {(pop >= 80).sum():,} ({(pop >= 80).mean():.2%})")

fig, ax = plt.subplots()
ax.hist(pop, bins=50, color="#4C72B0", edgecolor="white", linewidth=0.4)
ax.axvline(pop.median(), color="#55A868", linewidth=2.5, label=f"mediana {pop.median():.0f}")
ax.axvline(pop.mean(), color="#C44E52", linewidth=2.5, linestyle="--", label=f"media {pop.mean():.2f}")
ax.set_title("El 8,2 % del catálogo tiene popularidad cero: ese montón arrastra la media por debajo de la mediana")
ax.set_xlabel("popularidad (0-100)")
ax.set_ylabel("número de canciones")
ax.legend()
plt.tight_layout()
plt.show()

📌 **Media 42,48 y mediana 45.** La asimetría no siempre infla: aquí hay 2 703 canciones con
popularidad exactamente cero —catálogo que nadie reproduce— y ese montón en el extremo izquierdo
tira la media hacia abajo. Quitando los ceros, la media sube a 46,29. Ninguna de las dos versiones
es «la correcta» por defecto: *«¿qué tan popular es lo que tenemos?»* y *«¿qué tan popular es lo que
suena?»* son dos preguntas distintas, y elegir una es una decisión de negocio que se escribe al lado
del número. La regla de la semana: **antes de elegir media o mediana, mira la forma**. La duración
tenía cola a la derecha; la popularidad, un pico en cero. El histograma tarda cuatro segundos y te
ahorra el ridículo.

## 6. Frecuencias: `value_counts`

Para las variables categóricas no hay media que valga. `value_counts()` da el conteo y
`value_counts(normalize=True)` la proporción, que es casi siempre lo que quieres reportar.

In [ ]:
conteo = canciones["playlist_genre"].value_counts()
proporcion = canciones["playlist_genre"].value_counts(normalize=True) * 100
pop_mediana = canciones.groupby("playlist_genre")["track_popularity"].median()

generos = pd.DataFrame({
    "filas": conteo,
    "% del catálogo": proporcion.round(1),
    "popularidad mediana": pop_mediana,
}).sort_values("filas", ascending=False)
generos

El género con más presencia —edm, 6 043 filas, el 18,4 %— es también el de la popularidad mediana
más baja (36). Y pop, con menos filas (5 507), tiene la mediana más alta (52). **Tener mucho de algo
y que ese algo funcione son dos indicadores distintos**, y la tabla los muestra juntos precisamente
para que nadie los confunda: es la versión musical del ranking engañoso de la semana 1.

## 7. Dispersión: desviación, rango intercuartílico y coeficiente de variación

El centro sin la dispersión es media información. La desviación estándar está en la unidad del dato
—minutos o puntos— y por eso no sirve para comparar variables distintas; el **coeficiente de
variación** (desviación dividida por media) no tiene unidades y sí.

In [ ]:
def dispersion(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    return pd.Series({"n": s.size, "media": s.mean(), "mediana": s.median(),
                      "desviación": s.std(), "IQR": q3 - q1, "CV": s.std() / s.mean()})

tabla = pd.DataFrame({
    "duración (min)": dispersion(d),
    "popularidad": dispersion(pop),
    "tempo (bpm)": dispersion(canciones["tempo"]),
})
tabla

📌 **CV de duración 0,27; CV de popularidad 0,59.** La duración de una canción es un fenómeno
disciplinado: la industria converge al formato de tres minutos y medio. La popularidad es más del
doble de dispersa: el éxito no tiene formato. Consecuencia práctica: puedes planificar una parrilla
de radio con la duración típica y equivocarte poco; no puedes planificar ingresos con la popularidad
típica, porque «típica» apenas existe.

In [ ]:
# ✅ Comprobación 1 · el perfilado descansa sobre estas cifras
assert len(canciones) == 32_833, \
    f"Deberías tener 32 833 filas y tienes {len(canciones):,}: ¿filtraste sin querer?"
assert canciones["track_id"].nunique() == 28_356, \
    "Deberían ser 28 356 canciones únicas: revisa que no hayas deduplicado todavía"
assert abs(d.mean() - 3.76) < 0.01 and abs(d.median() - 3.60) < 0.01, \
    "Media o mediana de duración fuera de lugar (3,76 y 3,60): ¿convertiste duration_ms entre 60 000?"

print(f"Comprobación 1 superada ✓  {len(canciones):,} filas, {canciones['track_id'].nunique():,} canciones, "
      f"duración media {d.mean():.2f} y mediana {d.median():.2f}")

### 🌶️ Ejercicio 1 — Guiado

Repite la sección 3 completa abriendo por **`playlist_genre`**: una tabla con media, mediana,
desviación y CV de la duración por género, y un boxplot que los compare. Después responde: ¿qué
género corre más largo? ¿En cuál se parecen menos sus canciones entre sí? ¿Alguno muestra señales de
tener dos poblaciones dentro?

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: canciones.groupby("playlist_genre")["duracion_min"].apply(dispersion).unstack()
# Pista 2: sns.boxplot(data=canciones, x="duracion_min", y="playlist_genre", showfliers=False)
# Pista 3: si en algún género la media y la mediana se separan más que en el resto,
#          dibuja su histograma antes de responder lo de las dos poblaciones.

### 🔥 Desafío

El `describe()` de la sección 2 dice que la canción más corta dura **0,07 minutos: cuatro
segundos**. Encuentra todas las «canciones» de menos de un minuto, cuenta cuántas son, revisa sus
nombres —vas a encontrar interludios, saludos y cortinillas— y decide, con argumento escrito, si
entran o no en la línea base de duración del catálogo. Cuantifica cuánto se mueve la media si las
excluyes y compara ese movimiento con el de la mediana.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: cortas = canciones[canciones["duracion_min"] < 1]
# Pista 2: mira cortas[["track_name", "track_artist", "duracion_min"]].sort_values("duracion_min")
# Pista 3: compara d.mean() con d[d >= 1].mean() y haz lo mismo con la mediana.
#          El contraste entre los dos movimientos ES la respuesta.

### 🎯 Reto en clase (15 min)

En equipos, contra reloj. Doce preguntas descriptivas sobre el catálogo de Spotify. Para **cada
respuesta** hay que indicar si el estadístico correcto es la media o la mediana y justificarlo con la
forma de la distribución, no con la costumbre.

1. ¿Cuántas filas tiene el archivo y cuántas canciones únicas contiene? ¿Por qué no coinciden?
2. ¿Cuánto dura una canción típica? ¿Reportas media o mediana, y por qué?
3. ¿Qué proporción del catálogo pasa de 5 minutos y qué proporción no llega a 2?
4. ¿Cuál es el género con más filas y qué porcentaje del catálogo concentra?
5. ¿Cuál es el rango intercuartílico de la duración y qué te dice de la mitad central?
6. ¿Qué género tiene la popularidad mediana más alta? ¿Cambia el ranking si usas la media?
7. ¿Cuántos artistas, ordenados de mayor a menor presencia, hacen falta para acumular el 50 % de las filas?
8. ¿Cuál es el CV de la duración por género? ¿Qué género es el más homogéneo?
9. ¿En qué percentil cae una canción de exactamente 5 minutos dentro de rock? ¿Y dentro de edm?
10. ¿Qué porcentaje de canciones tiene popularidad 0 y cómo se reparte entre géneros?
11. Compara la popularidad de las canciones que aparecen en una sola playlist contra las que
    aparecen en dos o más. ¿Qué implica para un «ranking de éxitos» calculado sobre filas?
12. `value_counts()` sobre `playlist_subgenre` da 24 valores. Si agrupas todo salvo el top 10 en
    «Otros», ¿qué porcentaje de las filas queda escondido en esa categoría? ¿Es aceptable?

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: una celda por pregunta, con la respuesta impresa y una línea de comentario
# que diga "media" o "mediana" y por qué. Las preguntas 7 y 11 son las que más tiempo toman.

## La trampa de hoy

⚠️ **Promediar filas cuando la unidad es otra cosa.** El sello reporta al directorio «la popularidad
media de nuestro catálogo es 42». El auditor, con el mismo archivo, calcula 39. Ninguno se equivocó
en el `mean()`: uno promedió apariciones en playlists y el otro canciones. Y como las canciones
populares aparecen en más playlists, cada copia extra es un voto extra para las famosas.

In [ ]:
por_fila = canciones["track_popularity"].mean()

unicas = canciones.sort_values("track_popularity", ascending=False).drop_duplicates("track_id")
por_cancion = unicas["track_popularity"].mean()

apariciones = canciones["track_id"].value_counts()
repetidas = canciones["track_id"].isin(apariciones[apariciones > 1].index)

print(f"popularidad media por fila (aparición) : {por_fila:.2f}")
print(f"popularidad media por canción única    : {por_cancion:.2f}")
print(f"diferencia                             : {por_fila - por_cancion:.2f} puntos\n")
print(f"canciones en dos o más playlists : {apariciones.gt(1).sum():,} "
      f"(hasta {apariciones.max()} apariciones la más repetida)")
print(f"popularidad media de las filas repetidas    : {canciones.loc[repetidas, 'track_popularity'].mean():.1f}")
print(f"popularidad media de las filas de una sola  : {canciones.loc[~repetidas, 'track_popularity'].mean():.1f}")

**42,48 por fila contra 39,33 por canción: 3,2 puntos de inflación** sin tocar una sola cifra del
archivo. Las 3 166 canciones que están en dos o más playlists promedian 60,2 de popularidad; las de
una sola, 37,1. La repetición no es ruido: es correlación pura entre éxito y presencia, y cualquier
promedio sobre filas la convierte en sesgo. Es la lección de la semana 2 mordiendo en la semana 3:
**la unidad de análisis decide a quién le das voz**, y aquí las famosas votaban hasta diez veces.

La media no está mal calculada: está calculada sobre otra pregunta. «¿Qué tan popular es lo que
colocamos en playlists?» → por fila. «¿Qué tan popular es nuestro catálogo?» → por canción. Elegir
sin decirlo es elegir el titular de la reunión.

In [ ]:
# ✅ Comprobación 2 · la trampa queda cuantificada con estas cifras
assert generos.loc["edm", "filas"] == 6_043 and generos.loc["rock", "filas"] == 4_951, \
    "El conteo por género no cuadra (edm 6 043, rock 4 951): revisa el value_counts sin filtros"
assert abs(por_fila - 42.48) < 0.01 and abs(por_cancion - 39.33) < 0.01, \
    "Las dos medias de popularidad deberían ser 42,48 (por fila) y 39,33 (por canción): revisa el drop_duplicates por track_id"

print("Comprobación 2 superada ✓  misma columna, mismo mean(), dos respuestas: la unidad manda")

## Entregable

Sube `lab_03_apellido.ipynb` con:

- El cuaderno ejecutado, con el histograma de duración y el de popularidad.
- Los tres ejercicios resueltos, incluido el argumento escrito del desafío sobre las canciones de
  menos de un minuto.
- Las doce preguntas respondidas, cada una con su estadístico elegido y una línea de justificación.
- Tres observaciones descriptivas que un gerente de catálogo podría accionar mañana, con la cifra
  que las respalda.
- Una fila nueva en la bitácora: pídele al asistente que genere doce preguntas descriptivas sobre tu
  propio dataset y verifica una por una la respuesta que te dio contra lo que devuelve pandas. Al
  menos una no va a coincidir; anota cuál y por qué.

## Para tu equipo

- El perfilado descriptivo del dataset del caso es el entregable de esta semana. Empiecen por
  `describe(include="all")` y anoten todo lo que llame la atención: mínimos imposibles, medias y
  medianas separadas, counts desiguales.
- Ninguna cifra del informe del grupo va sola: media o mediana **siempre** acompañada de una medida
  de dispersión y del número de observaciones que la sostienen.
- Revisen si en su dataset una fila es la unidad que creen: si hay repetición (un cliente en varias
  campañas, un producto en varias tiendas), el promedio por fila ya viene sesgado de fábrica, como la
  popularidad de hoy.